# 04 — Baseline LSTM Training

This notebook runs `src/train_LSTM_baseline.py`, which executes the full Phase 3a pipeline:

1. Load the train / val / test splits from `data/processed/`.
2. Select input feature columns (configurable from the CLI).
3. Log-transform the target `realized_vol_21d` for training.
4. Fit a `StandardScaler` on the train features.
5. Build sliding-window `DataLoader`s.
6. Run an **Optuna** study (search space in `config.LSTM_SEARCH_SPACE`) — each trial is scored by validation MSE back in the *original* (inverse-log) volatility scale.
7. Retrain the model with the best hyperparameters and evaluate it on the test set (MSE / RMSE / MAE, original scale).
8. Save artifacts to `models/lstm_baseline.pt` and `models/lstm_baseline_scaler.joblib`.

The defaults match the feature list requested for the baseline: `Open, High, Low, Close, Volume, log_return, abs_return, oc_return, intraday_range, log_volume`. Pass `--features ...` to override.

In [1]:
import json
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
SCRIPT = REPO_ROOT / "src" / "train_LSTM_baseline.py"
assert SCRIPT.exists(), f"Missing: {SCRIPT}"
print("Repo root:", REPO_ROOT)
print("Script   :", SCRIPT)

Repo root: /Users/maharajhaider/gitrepo/StockVolatilitySight
Script   : /Users/maharajhaider/gitrepo/StockVolatilitySight/src/train_LSTM_baseline.py


## Run the training pipeline

Tweak `ARGS` below to change the trial count, epochs, or feature list. The defaults come from `config.py` (`LSTM_N_TRIALS`, `LSTM_TUNE_EPOCHS`, `LSTM_FINAL_EPOCHS`, `LSTM_BASELINE_FEATURES`).

Output is streamed line-by-line from the subprocess so you can watch each epoch log as it happens. `stderr` is merged into `stdout` (the training script logs progress via `logging`, which defaults to stderr — that's why you weren't seeing anything before).

In [2]:
ARGS = [
    # "--features", "Close", "Volume", "log_return", "abs_return", "log_volume",
    # "--n-trials", "20",
    # "--tune-epochs", "30",
    # "--final-epochs", "80",
]

# -u = unbuffered Python stdout/stderr so log lines reach us immediately.
cmd = [sys.executable, "-u", str(SCRIPT), *ARGS]
print("Running:", " ".join(cmd))
print("-" * 80)

# Merge stderr into stdout so `logger.info(...)` lines stream alongside prints.
# bufsize=1 + text=True gives line-buffered reads from the pipe.
proc = subprocess.Popen(
    cmd,
    cwd=str(REPO_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

captured_lines = []
assert proc.stdout is not None
try:
    for line in proc.stdout:
        print(line, end="")          # live feed into the notebook
        captured_lines.append(line)  # keep for downstream JSON parsing
finally:
    return_code = proc.wait()

combined_output = "".join(captured_lines)
if return_code != 0:
    raise RuntimeError(f"train_LSTM_baseline.py exited with code {return_code}")

Running: /Users/maharajhaider/gitrepo/StockVolatilitySight/venv/bin/python3.13 -u /Users/maharajhaider/gitrepo/StockVolatilitySight/src/train_LSTM_baseline.py
--------------------------------------------------------------------------------


20:58:55 | INFO    | train_LSTM_baseline | Features: ['log_return', 'abs_return', 'oc_return', 'intraday_range', 'relative_volume_21d']
20:58:55 | INFO    | train_LSTM_baseline | Target  : realized_vol_21d
20:58:55 | INFO    | train_LSTM_baseline | Split sizes — train=3020, val=1089, test=1475
20:58:55 | INFO    | train_LSTM_baseline | n_features=5 | rows — train=3001 val=1089 test=1475
20:58:55 | INFO    | train_LSTM_baseline | Starting Optuna study with 20 trials…


20:58:56 | INFO    | train_LSTM_baseline | epoch   1 | train_log_mse=3.659263 | val_mse_raw=0.00004915 | best=0.00004915


20:58:56 | INFO    | train_LSTM_baseline | epoch   2 | train_log_mse=0.165376 | val_mse_raw=0.00005729 | best=0.00004915


20:58:57 | INFO    | train_LSTM_baseline | epoch   3 | train_log_mse=0.140873 | val_mse_raw=0.00004866 | best=0.00004866


...

21:11:03 | INFO    | train_LSTM_baseline | epoch  23 | train_log_mse=0.122554 | val_mse_raw=0.00004442 | best=0.00003946


21:11:05 | INFO    | train_LSTM_baseline | epoch  24 | train_log_mse=0.119747 | val_mse_raw=0.00004363 | best=0.00003946


21:11:08 | INFO    | train_LSTM_baseline | epoch  25 | train_log_mse=0.124737 | val_mse_raw=0.00004266 | best=0.00003946


21:11:11 | INFO    | train_LSTM_baseline | epoch  26 | train_log_mse=0.108674 | val_mse_raw=0.00004267 | best=0.00003946
21:11:11 | INFO    | train_LSTM_baseline | Early stopping at epoch 26
21:11:11 | INFO    | train_LSTM_baseline | Final retrain — best val MSE (raw scale): 0.00003946


21:11:12 | INFO    | train_LSTM_baseline | Test metrics: {'MSE': 1.5175318367255386e-05, 'RMSE': 0.003895551199093461, 'MAE': 0.00246306206099689, 'n_test_windows': 1434}
21:11:12 | INFO    | train_LSTM_baseline | Saved model → /Users/maharajhaider/gitrepo/StockVolatilitySight/models/lstm_baseline.pt
21:11:12 | INFO    | train_LSTM_baseline | Saved scaler → /Users/maharajhaider/gitrepo/StockVolatilitySight/models/lstm_baseline_scaler.joblib

=== Baseline LSTM results ===
{
  "best_params": {
    "hidden_size": 128,
    "n_layers": 2,
    "dropout": 0.03252579649263976,
    "lr": 0.007902619549708232,
    "batch_size": 32,
    "seq_len": 42
  },
  "best_val_mse_raw": 3.9455007936339825e-05,
  "retrained_val_mse_raw": 3.9455007936339825e-05,
  "test_metrics": {
    "MSE": 1.5175318367255386e-05,
    "RMSE": 0.003895551199093461,
    "MAE": 0.00246306206099689,
    "n_test_windows": 1434
  },
  "features": [
    "log_return",
    "abs_return",
    "oc_return",
    "intraday_range",
    "r

## Parse the results

The script prints a JSON block under `=== Baseline LSTM results ===`. We extract it here for easy display in downstream analysis.

In [3]:
marker = "=== Baseline LSTM results ==="
idx = combined_output.find(marker)
assert idx != -1, "Results marker not found in script output."
json_blob = combined_output[idx + len(marker):].strip()
results = json.loads(json_blob)

print("Best hyperparameters:")
for k, v in results["best_params"].items():
    print(f"  {k:>12}: {v}")

print(f"\nValidation MSE (raw scale, Optuna best) : {results['best_val_mse_raw']:.8f}")
print(f"Validation MSE (raw scale, final retrain): {results['retrained_val_mse_raw']:.8f}")

print("\nTest metrics (raw scale):")
for k, v in results["test_metrics"].items():
    print(f"  {k:>6}: {v}")

print(f"\nFeatures used ({len(results['features'])}): {results['features']}")
print(f"Target        : {results['target']}")
print(f"N trials      : {results['n_trials']}")

Best hyperparameters:
   hidden_size: 128
      n_layers: 2
       dropout: 0.03252579649263976
            lr: 0.007902619549708232
    batch_size: 32
       seq_len: 42

Validation MSE (raw scale, Optuna best) : 0.00003946
Validation MSE (raw scale, final retrain): 0.00003946

Test metrics (raw scale):
     MSE: 1.5175318367255386e-05
    RMSE: 0.003895551199093461
     MAE: 0.00246306206099689
  n_test_windows: 1434

Features used (5): ['log_return', 'abs_return', 'oc_return', 'intraday_range', 'relative_volume_21d']
Target        : realized_vol_21d
N trials      : 20


## Saved artifacts

- `models/lstm_baseline.pt` — model `state_dict`, selected hyperparameters, feature list.
- `models/lstm_baseline_scaler.joblib` — the `StandardScaler` fit on the train features (needed to reproduce predictions).

Reload example:

```python
import torch, joblib
from src.lstm_model import LSTMRegressor
ckpt = torch.load('../models/lstm_baseline.pt', weights_only=False)
model = LSTMRegressor(input_size=ckpt['n_features'], **{k: ckpt['hyperparameters'][k] for k in ('hidden_size','n_layers','dropout')})
model.load_state_dict(ckpt['state_dict'])
scaler = joblib.load('../models/lstm_baseline_scaler.joblib')
```